# SIV: Systematic Inference-Based Verification for NL-to-FOL Translation

End-to-end pipeline notebook.  Works on Google Colab and locally.

**Stages:**
1. Symbolic Pre-Analysis (Stage 1)
2. LLM Extraction with enriched prompts (Stage 2)
3. Deterministic FOL test compilation (Stage 3)
4. Tiered verification with partial credit
5. SIV Score leaderboard

In [ ]:
# ── Cell 1: Setup & Installation ─────────────────────────────────────────────
import sys, os

# Install dependencies (quiet on Colab; skip locally if already installed)
!pip install -q transformers datasets torch nltk sentencepiece accelerate openai pandas spacy tqdm
!python -m spacy download en_core_web_sm -q

import nltk
for resource in ['wordnet', 'averaged_perceptron_tagger_eng', 'punkt_tab', 'brown']:
    nltk.download(resource, quiet=True)

# Mount Google Drive / add repo root to path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/siv-project'  # adjust if needed
except ImportError:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f'Repository root: {REPO_ROOT}')

In [ ]:
# ── Cell 2: API key setup ─────────────────────────────────────────────────────
import os

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY') or ''
except Exception:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

if OPENAI_API_KEY:
    import openai
    client = openai.OpenAI(api_key=OPENAI_API_KEY)
    USE_API = True
    print('OpenAI API available — will use GPT-4o for extraction.')
else:
    client = None
    USE_API = False
    print('No API key — falling back to rule-based extraction.')

In [ ]:
# ── Cell 3: Import SIV modules ────────────────────────────────────────────────
from siv.schema import (
    Entity, EntityType, Fact, MacroTemplate,
    ProblemExtraction, SentenceExtraction,
    UnitTest, TestSuite, VerificationResult,
)
from siv.pre_analyzer import analyze_sentence, format_analyses_for_prompt
from siv.extractor import extract_problem, _fallback_extraction
from siv.compiler import compile_test_suite
from siv.verifier import verify
from siv.scorer import compute_siv_score, score_candidates, aggregate_scores, macro_average
from siv.fol_utils import is_valid_fol, extract_predicates, NLTK_AVAILABLE
from siv.vampire_interface import is_vampire_available, setup_vampire

print(f'NLTK FOL parser available: {NLTK_AVAILABLE}')
print(f'Vampire prover available:  {is_vampire_available()}')

In [ ]:
# ── Cell 4: (Optional) Install Vampire ────────────────────────────────────────
# Uncomment if running on Colab and Vampire is not yet installed:
# from siv.vampire_interface import setup_vampire
# setup_vampire(target_dir=REPO_ROOT)

In [ ]:
# ── Cell 5: Load FOLIO dataset ────────────────────────────────────────────────
import json, os
from pathlib import Path

DATA_DIR = Path(REPO_ROOT) / 'data'
folio_path = DATA_DIR / 'folio_problems.json'

if folio_path.exists():
    with open(folio_path) as f:
        folio_problems = json.load(f)
    print(f'Loaded {len(folio_problems)} FOLIO problems from {folio_path}')
else:
    # Download from HuggingFace
    print('folio_problems.json not found — downloading from HuggingFace...')
    from datasets import load_dataset
    ds = load_dataset('yale-nlp/FOLIO', split='validation')
    # Group by story_id
    problems = {}
    for row in ds:
        pid = row.get('story_id', row.get('id', 'unknown'))
        if pid not in problems:
            problems[pid] = {'premises': [], 'hypothesis': '', 'label': ''}
        if row.get('sentence_type') == 'premise':
            problems[pid]['premises'].append(row['sentence'])
        else:
            problems[pid]['hypothesis'] = row.get('sentence', '')
            problems[pid]['label'] = row.get('label', '')
    folio_problems = list(problems.values())[:100]   # first 100 for demo
    DATA_DIR.mkdir(exist_ok=True)
    with open(folio_path, 'w') as f:
        json.dump(folio_problems, f, indent=2)
    print(f'Saved {len(folio_problems)} problems to {folio_path}')

In [ ]:
# ── Cell 6: Stage 1 Demo — Pre-Analysis ──────────────────────────────────────
demo_sentences = [
    'The tall red tree grows quickly.',
    'A Harvard student passed the exam.',
    'Lana Wilson directed After Tiller.',
    'All professional athletes train daily.',
]

print('=' * 60)
print('STAGE 1: Symbolic Pre-Analysis')
print('=' * 60)
for sent in demo_sentences:
    analyses = analyze_sentence(sent)
    print(f'\nSentence: "{sent}"')
    if analyses:
        for ca in analyses:
            print(f'  {ca.modifier!r} + {ca.noun!r}: {ca.recommendation} ({ca.reason})')
    else:
        print('  (no modifier+noun compounds detected)')

In [ ]:
# ── Cell 7: Stage 2 Demo — LLM / Fallback Extraction ─────────────────────────
print('=' * 60)
print('STAGE 2: Entity + Fact Extraction')
print('=' * 60)

for sent in demo_sentences[:3]:
    analyses = analyze_sentence(sent)
    extraction = _fallback_extraction(sent, analyses)   # use fallback for demo
    print(f'\nSentence: "{sent}"')
    print(f'  Macro template: {extraction.macro_template.value}')
    print(f'  Entities: {[(e.id, e.surface, e.entity_type.value) for e in extraction.entities]}')
    print(f'  Facts:    {[(f.pred, f.args, f.negated) for f in extraction.facts]}')

In [ ]:
# ── Cell 8: Stage 3 Demo — Compile Unit Tests ─────────────────────────────────
from siv.extractor import extract_problem as ep

demo_problem_sentences = [
    'The crimson car is running.',
    'All cars have engines.',
]

problem_extraction = ep(
    demo_problem_sentences,
    client=client,
    use_api=USE_API,
    problem_id='demo_car',
)

test_suite = compile_test_suite(problem_extraction)

print('=' * 60)
print('STAGE 3: Compiled Unit Tests')
print('=' * 60)
print(f'\nTotal tests: {test_suite.total_tests} '
      f'({len(test_suite.positive_tests)} recall + '
      f'{len(test_suite.negative_tests)} precision)')

print('\n[+] RECALL tests (candidate must entail):')
for t in test_suite.positive_tests:
    valid = is_valid_fol(t.fol_string)
    print(f'  [{t.test_type:12s}] {t.fol_string}  {"✓" if valid else "✗ INVALID"}')

print('\n[-] PRECISION tests (candidate must NOT entail):')
for t in test_suite.negative_tests:
    valid = is_valid_fol(t.fol_string)
    print(f'  [{t.test_type:12s}] {t.fol_string}  {"✓" if valid else "✗ INVALID"}')

In [ ]:
# ── Cell 9: Candidate Evaluation Demo ────────────────────────────────────────
candidates = [
    'exists x.(Car(x) & Crimson(x) & Running(x))',   # good
    'exists x.Car(x)',                                # missing properties
    'all x.(Car(x) -> CrimsonRunning(x))',            # slugged predicate
    'exists x Car(x',                                 # invalid syntax
]

print('=' * 60)
print('VERIFICATION: Scoring Candidates')
print('=' * 60)

scored = score_candidates(candidates, test_suite)
for rank, cs in enumerate(scored, 1):
    print(f'\n#{rank}: "{cs.candidate_fol[:70]}"')
    print(f'  SIV={cs.siv_score:.3f}  '
          f'recall={cs.recall_rate:.2f}  '
          f'precision={cs.precision_rate:.2f}  '
          f'syntax={cs.syntax_valid}')

In [ ]:
# ── Cell 10: T5 Candidate Generation (requires pretrained model) ──────────────
# Uncomment and adapt to load your T5 checkpoint.
#
# from transformers import T5ForConditionalGeneration, T5Tokenizer
# import torch
#
# MODEL_NAME = 'google/flan-t5-base'   # or your MALLS warmup checkpoint
# tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
# model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
# model.eval()
#
# def generate_candidates(sentence, num_beams=5, max_new_tokens=128):
#     input_ids = tokenizer(sentence, return_tensors='pt').input_ids
#     with torch.no_grad():
#         outputs = model.generate(
#             input_ids,
#             num_beams=num_beams,
#             num_return_sequences=num_beams,
#             max_new_tokens=max_new_tokens,
#         )
#     return [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
#
# for sent in demo_problem_sentences:
#     candidates = generate_candidates(sent)
#     scores = score_candidates(candidates, test_suite)
#     print(f'\n"{sent}"  → best SIV: {scores[0].siv_score:.3f}')
print('(T5 generation cell — uncomment and configure a checkpoint to use.)')

In [ ]:
# ── Cell 11: Full FOLIO Evaluation Loop ───────────────────────────────────────
import pandas as pd
from tqdm.auto import tqdm

# Run on first N problems for demonstration
N_PROBLEMS = 10
EVAL_PROBLEMS = folio_problems[:N_PROBLEMS]

problem_results = {}

for prob in tqdm(EVAL_PROBLEMS, desc='Evaluating FOLIO problems'):
    pid = prob.get('id', prob.get('story_id', 'unknown'))
    premises = prob.get('premises', [])
    if not premises:
        continue

    # Stage 1 + 2: extract
    extraction = ep(premises, client=client, use_api=USE_API, problem_id=pid)

    # Stage 3: compile test suite
    suite = compile_test_suite(extraction)

    # Use hypothesis FOL as the single candidate (if available)
    gold_fol = prob.get('hypothesis_fol', prob.get('hypothesis', ''))
    candidates_to_score = [gold_fol] if gold_fol else []

    problem_results[pid] = score_candidates(candidates_to_score, suite) if candidates_to_score else []

print(f'Evaluated {len(problem_results)} problems.')

In [ ]:
# ── Cell 12: Results Leaderboard ──────────────────────────────────────────────
agg = aggregate_scores(problem_results)
avg = macro_average(agg)

rows = []
for ps in agg:
    rows.append({
        'problem_id':   ps.problem_id,
        'siv_score':    round(ps.best_siv_score, 3),
        'recall':       round(ps.best_recall, 3),
        'precision':    round(ps.best_precision, 3),
        'n_candidates': ps.num_candidates,
        'best_fol':     ps.best_candidate[:80],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

print(f'\nMacro-average  SIV={avg["siv"]:.3f}  '
      f'recall={avg["recall"]:.3f}  '
      f'precision={avg["precision"]:.3f}')

In [ ]:
# ── Cell 13: Export Results ───────────────────────────────────────────────────
out_path = Path(REPO_ROOT) / 'results_siv.csv'
df.to_csv(out_path, index=False)
print(f'Results saved to {out_path}')